In [1]:
from mpramnist.deAlmeida2022.dataset import deAlmeidaDataset
from mpramnist.deAlmeida2022.trainer import LitModel_deAlmeida

from mpramnist.models import DeepStarr

from mpramnist import transforms as t

import torch
import torch.nn as nn
import torch.utils.data as data

import lightning.pytorch as L

from torchmetrics import PearsonCorrCoef

BATCH_SIZE = 1024
NUM_WORKERS = 103

promoter_types = deAlmeidaDataset.CELL_TYPES # ["Dev_log2", "Hk_log2"]

# Reverse-complement

In case you want to use reverse-complement as it was used in the original study(*Dataset size: 2N (N original + N reverse-complemented sequences)*), then use *use_original_reverse_complement* attribute (Default True). Reverse-complement will be added in case if you use `split = 'train'`.

To turn off reverse-complement with training set use `use_original_reverse_complement = False, split = 'train'`

**WARNING**: in the original study, reverse-complement  was applied **only** to the training set.

For example:

In [2]:
train_transform = t.Compose([t.Seq2Tensor()])
val_test_transform = t.Compose([t.Seq2Tensor(),])

orig_rev_comp_train_dataset = deAlmeidaDataset(cell_type=promoter_types,split="train",transform=train_transform,root="../data/",)
val_dataset = deAlmeidaDataset(cell_type=promoter_types,split="val",transform=val_test_transform,root="../data/",)
test_dataset = deAlmeidaDataset(cell_type=promoter_types,split="test",transform=val_test_transform,root="../data/",)

print(len(orig_rev_comp_train_dataset),len(val_dataset),len(test_dataset))

/home/nios/MPRA-MNIST/mpramnist/deAlmeida2022/dataset.py:210: UserWarning: WARNING! 
Note: The training set contains reverse-complement augmentation as implemented in the original study.  
• Dataset size: 2N (N original + N reverse-complemented sequences)  
• Label consistency: y_rc ≡ y_original  
• Do not reapply this transformation during preprocessing. 
  warnings.warn(


402278 40570 41186


But we suggest using reverse-complement transformation by writing **transforms.Reverse(prob = 0.5)**. *prob = 0.5* means that the sequence will be reversed with probability of 0.5. So use `use_original_reverse_complement = False, split = 'train'`

For example:

In [3]:
train_transform = t.Compose([t.ReverseComplement(0.5),t.Seq2Tensor(),])
val_test_transform = t.Compose([t.ReverseComplement(0),t.Seq2Tensor(), ])

train_dataset = deAlmeidaDataset(cell_type=promoter_types,use_original_reverse_complement=False,split="train",transform=train_transform,root="../data/",)
val_dataset = deAlmeidaDataset(cell_type=promoter_types,split="val",transform=val_test_transform,root="../data/",)
test_dataset = deAlmeidaDataset(cell_type=promoter_types,split="test",transform=val_test_transform,root="../data/",)

print(len(train_dataset),len(val_dataset),len(test_dataset))

201139 40570 41186


# Data splitting

Sequences from the **first** and **second** half of chr2R were held out for validation and testing, respectively.

The remainig chromosomes are used for training set.

You can use *"train"*, *"val"*, *"test"* to define the training, validation or test set respectively **using the same approach as the original study**

For example:

In [4]:
train_transform = t.Compose([t.ReverseComplement(0.5),t.Seq2Tensor(),])
val_test_transform = t.Compose([t.ReverseComplement(0),t.Seq2Tensor(), ])

train_dataset = deAlmeidaDataset(cell_type=promoter_types,use_original_reverse_complement=False,split="train",transform=train_transform,root="../data/",)
val_dataset = deAlmeidaDataset(cell_type=promoter_types,split="val",transform=val_test_transform,root="../data/",)
test_dataset = deAlmeidaDataset(cell_type=promoter_types,split="test",transform=val_test_transform,root="../data/",)

print(len(train_dataset),len(val_dataset),len(test_dataset))

201139 40570 41186


## On the other hand, 

you can define a list of specific chromosomes that you want to use as training, validation ot test set

For example:

In [5]:
list_of_chr = deAlmeidaDataset.LIST_OF_CHR
print(list_of_chr)

['2L', '2LHet', '2RHet', '3L', '3LHet', '3R', '3RHet', '4', 'X', 'XHet', 'YHet', '2R']


In [6]:
my_train_dataset = deAlmeidaDataset(
    cell_type=promoter_types,
    split=[
        "2L",
        "2LHet",
        "2RHet",  # Reverse complement transformation is disabled for chromosome list splits.
        "3L",
        "3LHet",
        "3R",  # Set use_original_reverse_complement=True to apply original paper augmentation.
        "3RHet",
        "4",
    ],
    use_original_reverse_complement=False,
    transform=train_transform,
    root="../data/",
)

my_val_dataset = deAlmeidaDataset(cell_type=promoter_types,split=["X", "XHet", "YHet"],transform=val_test_transform,root="../data/",)
my_test_dataset = deAlmeidaDataset(cell_type=promoter_types,split="2R",transform=val_test_transform,root="../data/",)

# encapsulate data into dataloader form
train_loader = data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = data.DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = data.DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print(len(my_train_dataset), len(my_val_dataset), len(my_test_dataset))

in_channels = len(train_dataset[0][0])
out_channels = len(promoter_types)

151641 49498 81756


For more information about genomic regions check Use_Genomic_Regions.ipynb

## Trainer

In [8]:
model = DeepStarr(out_channels)
""" OR
from mpramnist.models import HumanLegNet
from mpramnist.models import initialize_weights

model = HumanLegNet(
    in_ch=in_channels,
    output_dim=out_channels,
    stem_ch=64,
    stem_ks=11,
    ef_ks=9,
    ef_block_sizes=[80, 96, 112, 128],
    pool_sizes=[2, 2, 2, 2],
    resize_factor=4,
)
model.apply(initialize_weights)"""

seq_model = LitModel_deAlmeida(
    model=model,
    loss=nn.MSELoss(),
    cell_types=promoter_types,
    weight_decay=1e-6,
    lr=2e-3,
    print_each=10,
)
# Initialize a trainer
trainer = L.Trainer(
    accelerator="gpu",
    devices=[1],
    max_epochs=5,
    gradient_clip_val=1,
    precision="16-mixed",
    enable_progress_bar=True,
    num_sanity_val_steps=0,
)
# Train the model
trainer.fit(seq_model, train_dataloaders=train_loader, val_dataloaders=val_loader)

Using 16bit Automatic Mixed Precision (AMP)
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/nios/miniconda3/envs/mpra/lib/python3.12/site-packages/lightning/pytorch/trainer/connectors/logger_connector/logger_connector.py:76: Starting from v1.9.0, `tensorboardX` has been removed as a dependency of the `lightning.pytorch` package, due to potential conflicts with other packages in the ML ecosystem. For this reason, `logger=True` will use `CSVLogger` as the default logger, unless the `tensorboard` or `tensorboardX` packages are found. Please `pip install lightning[extra]` or one of them to enable TensorBoard support by default
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly

Epoch 4: 100%|██████████| 197/197 [00:17<00:00, 11.44it/s, v_num=1, train_loss_step=1.460, val_loss=1.440, val_Dev_log2_pearson=0.605, val_Hk_log2_pearson=0.712, val_pearson=0.658, train_loss_epoch=1.410]

`Trainer.fit` stopped: `max_epochs=5` reached.


Epoch 4: 100%|██████████| 197/197 [00:17<00:00, 11.41it/s, v_num=1, train_loss_step=1.460, val_loss=1.440, val_Dev_log2_pearson=0.605, val_Hk_log2_pearson=0.712, val_pearson=0.658, train_loss_epoch=1.410]


In [9]:
forw_transform = t.Compose([t.Seq2Tensor()])
rev_transform = t.Compose([t.ReverseComplement(1),t.Seq2Tensor(),])

test_forw = deAlmeidaDataset(cell_type=promoter_types,split="test",transform=forw_transform,root="../data/",)
test_rev = deAlmeidaDataset(cell_type=promoter_types,split="test",transform=rev_transform,root="../data/",)

forw = data.DataLoader(dataset=test_forw,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)
rev = data.DataLoader(dataset=test_rev,batch_size=BATCH_SIZE,shuffle=False,num_workers=NUM_WORKERS,pin_memory=True,)

predictions_forw = trainer.predict(seq_model, dataloaders=forw)
targets = torch.cat([pred["target"] for pred in predictions_forw])
y_preds_forw = torch.cat([pred["predicted"] for pred in predictions_forw])

predictions_rev = trainer.predict(seq_model, dataloaders=rev)
y_preds_rev = torch.cat([pred["predicted"] for pred in predictions_rev])

mean_forw = torch.mean(torch.stack([y_preds_forw, y_preds_rev]), dim=0)

pears = PearsonCorrCoef(num_outputs=len(promoter_types))
print(promoter_types, " Pearson correlation")

print(pears(mean_forw, targets))

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 41/41 [00:00<00:00, 157.35it/s]


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting DataLoader 0: 100%|██████████| 41/41 [00:00<00:00, 205.59it/s]
['Dev_log2', 'Hk_log2']  Pearson correlation
tensor([0.6283, 0.7321])
